# Sparse, Dense & Hybrid Retrieval with Pyversity

Pyversity is **embedding-agnostic**: its diversification algorithms only care about pairwise similarities between vectors, not how those vectors were produced. This means it works equally well with:

- **Sparse** vectors from BM25 (keyword-based term frequencies)
- **Dense** vectors from fast static models like [potion-base-32M](https://huggingface.co/minishlab/potion-base-32M)
- **Hybrid** combinations of both

This notebook demonstrates all three paradigms on the same corpus and query, so you can see the difference each retrieval approach makes — and how pyversity adds diversity on top of each one.

In [1]:
%pip install pyversity bm25s sentence-transformers

/Users/thomasvandongen/Projects/oss/pyversity/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import scipy.sparse as sp
import bm25s
from sentence_transformers import SentenceTransformer
from pyversity import diversify, Strategy

/Users/thomasvandongen/Projects/oss/pyversity/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## The Corpus

We use 25 short passages spread across 5 ML topic clusters:

1. **Gradient descent / optimization** (passages 0–4)
2. **Neural networks / deep learning** (passages 5–9)
3. **Transformers / attention** (passages 10–14)
4. **Decision trees / random forests** (passages 15–19)
5. **Clustering algorithms** (passages 20–24)

**Query:** `"gradient descent optimization"`

Without diversification, the top results are dominated by gradient-descent passages — exactly the redundancy problem pyversity is designed to solve.

In [3]:
corpus = [
    # --- Gradient descent / optimization (0-4) ---
    "Gradient descent is an iterative optimization algorithm that minimizes a loss function by moving in the direction of the negative gradient.",
    "Stochastic gradient descent updates model parameters using a single random sample at each step, making it faster but noisier than batch gradient descent.",
    "Momentum accelerates gradient descent by accumulating a velocity vector in directions of persistent reduction, helping to escape local minima.",
    "The Adam optimizer combines adaptive learning rates with momentum, making it one of the most popular choices for training deep neural networks.",
    "Learning rate scheduling reduces the step size during training, allowing gradient descent to converge more reliably to a good minimum.",

    # --- Neural networks / deep learning (5-9) ---
    "A neural network consists of layers of interconnected nodes that learn hierarchical representations from data through backpropagation.",
    "Convolutional neural networks use shared weight filters to detect local patterns in images, dramatically reducing the number of parameters needed.",
    "Dropout regularization randomly deactivates neurons during training, preventing the network from over-relying on any single pathway.",
    "Batch normalization normalizes layer inputs to have zero mean and unit variance, stabilizing training and allowing higher learning rates.",
    "Residual connections let gradients flow directly through skip connections, enabling training of very deep networks without vanishing gradients.",

    # --- Transformers / attention (10-14) ---
    "The transformer architecture uses self-attention to weigh the importance of each token relative to all others in a sequence.",
    "Multi-head attention allows the model to attend to different representation subspaces simultaneously, capturing richer contextual relationships.",
    "Positional encodings inject information about token order into the transformer, since self-attention is permutation-invariant by default.",
    "BERT pre-trains a bidirectional transformer on masked language modeling, producing contextual embeddings used in many downstream tasks.",
    "The scaled dot-product attention divides queries and keys by the square root of their dimension to stabilize gradients during training.",

    # --- Decision trees / random forests (15-19) ---
    "A decision tree recursively partitions the feature space by selecting splits that maximize information gain or reduce Gini impurity.",
    "Random forests aggregate predictions from many decorrelated decision trees, reducing variance compared to a single tree.",
    "Boosting builds an ensemble of weak learners sequentially, each correcting the errors of the previous one to reduce bias.",
    "Feature importance in tree-based models is measured by how much each feature reduces impurity across all splits.",
    "Pruning a decision tree removes branches that provide little predictive power, improving generalization on unseen data.",

    # --- Clustering algorithms (20-24) ---
    "K-means clustering assigns each point to the nearest centroid and iteratively updates centroids until convergence.",
    "DBSCAN identifies clusters as dense regions separated by low-density areas, handling arbitrary shapes and marking outliers as noise.",
    "Hierarchical clustering builds a dendrogram by successively merging the two closest clusters until all points belong to one group.",
    "The silhouette score measures how similar a point is to its own cluster compared to other clusters, ranging from -1 to 1.",
    "Gaussian mixture models assume data is generated from a mixture of Gaussians and use the EM algorithm to estimate their parameters.",
]

topic_labels = (
    ["Gradient descent"] * 5
    + ["Neural networks"] * 5
    + ["Transformers"] * 5
    + ["Decision trees"] * 5
    + ["Clustering"] * 5
)

query = "gradient descent optimization"
K = 10   # candidates to retrieve
k = 5    # items to select after diversification

print(f"Corpus size : {len(corpus)} passages")
print(f"Query       : '{query}'")
print(f"Retrieve top-{K}, then diversify to {k}")

Corpus size : 25 passages
Query       : 'gradient descent optimization'
Retrieve top-10, then diversify to 5


---
## Part 1: Sparse Retrieval with BM25

BM25 is a classic keyword-matching algorithm that scores documents based on term frequency and inverse document frequency, producing **sparse** term vectors — most entries are zero because most words do not appear in a given document.

Pyversity only needs an array of shape `(n_candidates, n_features)`. For BM25, we reconstruct the internal score matrix and slice to the top-K candidates. A single `.toarray()` call converts the scipy sparse matrix to a dense NumPy array that pyversity can consume.

> The primary point of this section is the API bridge: pyversity works with any NumPy array, sparse or dense. The diversity effect here is modest — BM25 term vectors capture vocabulary overlap rather than semantic similarity, so they are a coarser signal for redundancy than dense embeddings.

In [4]:
# Index the corpus with BM25
corpus_tokens = bm25s.tokenize(corpus)
retriever = bm25s.BM25()
retriever.index(corpus_tokens)

# Retrieve top-K candidates
query_tokens = bm25s.tokenize([query])
results, scores = retriever.retrieve(query_tokens, corpus=corpus, k=K)

bm25_candidate_texts   = list(results[0])   # top-K passage texts
bm25_candidate_indices = [corpus.index(t) for t in bm25_candidate_texts]
bm25_candidate_scores  = scores[0]           # BM25 relevance scores

print("Top-K BM25 candidates:")
for rank, (idx, score) in enumerate(zip(bm25_candidate_indices, bm25_candidate_scores), 1):
    print(f"  {rank}. [{topic_labels[idx]:>17s}] (score={score:.3f}) {corpus[idx][:70]}...")

Top-K BM25 candidates:
  1. [ Gradient descent] (score=3.060) Gradient descent is an iterative optimization algorithm that minimizes...
  2. [ Gradient descent] (score=1.824) Stochastic gradient descent updates model parameters using a single ra...
  3. [ Gradient descent] (score=1.426) Momentum accelerates gradient descent by accumulating a velocity vecto...
  4. [ Gradient descent] (score=1.342) Learning rate scheduling reduces the step size during training, allowi...
  5. [ Gradient descent] (score=0.000) The Adam optimizer combines adaptive learning rates with momentum, mak...
  6. [  Neural networks] (score=0.000) A neural network consists of layers of interconnected nodes that learn...
  7. [  Neural networks] (score=0.000) Convolutional neural networks use shared weight filters to detect loca...
  8. [  Neural networks] (score=0.000) Dropout regularization randomly deactivates neurons during training, p...
  9. [  Neural networks] (score=0.000) Batch normalization normalizes lay

In [5]:
# bm25s stores its term scores as raw CSR/CSC arrays.
# We reassemble them into a scipy sparse matrix (docs × vocab),
# then slice to our K candidates and call .toarray() —
# the only step needed to bridge sparse BM25 vectors into pyversity.
s = retriever.scores
bm25_score_matrix = sp.csc_matrix(
    (np.array(s["data"]), np.array(s["indices"]), np.array(s["indptr"])),
    shape=(int(s["num_docs"]), len(s["indptr"]) - 1),
)  # shape: (n_docs, vocab_size)

bm25_candidate_embeddings = bm25_score_matrix[bm25_candidate_indices].toarray()  # (K, vocab_size)

print(f"Sparse BM25 matrix shape : {bm25_score_matrix.shape}")
print(f"Candidate embeddings     : {bm25_candidate_embeddings.shape}  (K × vocab_size)")
print(f"Sparsity                 : {(bm25_candidate_embeddings == 0).mean():.1%} zeros")

Sparse BM25 matrix shape : (25, 270)
Candidate embeddings     : (10, 270)  (K × vocab_size)
Sparsity                 : 94.5% zeros


In [6]:
# Naive top-5: diversity=0.0 → pure relevance ranking, no diversification
naive_bm25 = diversify(
    embeddings=bm25_candidate_embeddings,
    scores=bm25_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.0,
)

print("=== BM25 — Naive top-5 (diversity=0.0) ===")
for rank, i in enumerate(naive_bm25.indices, 1):
    idx = bm25_candidate_indices[i]
    print(f"  {rank}. [{topic_labels[idx]:>17s}] {corpus[idx][:80]}...")

=== BM25 — Naive top-5 (diversity=0.0) ===
  1. [ Gradient descent] Gradient descent is an iterative optimization algorithm that minimizes a loss fu...
  2. [ Gradient descent] Stochastic gradient descent updates model parameters using a single random sampl...
  3. [ Gradient descent] Momentum accelerates gradient descent by accumulating a velocity vector in direc...
  4. [ Gradient descent] Learning rate scheduling reduces the step size during training, allowing gradien...
  5. [ Gradient descent] The Adam optimizer combines adaptive learning rates with momentum, making it one...


In [7]:
# Diversified top-5 with DPP
diverse_bm25 = diversify(
    embeddings=bm25_candidate_embeddings,
    scores=bm25_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.7,
)

print("=== BM25 — Diversified top-5 (diversity=0.7) ===")
for rank, i in enumerate(diverse_bm25.indices, 1):
    idx = bm25_candidate_indices[i]
    print(f"  {rank}. [{topic_labels[idx]:>17s}] {corpus[idx][:80]}...")

=== BM25 — Diversified top-5 (diversity=0.7) ===
  1. [ Gradient descent] Gradient descent is an iterative optimization algorithm that minimizes a loss fu...
  2. [ Gradient descent] Stochastic gradient descent updates model parameters using a single random sampl...
  3. [ Gradient descent] Momentum accelerates gradient descent by accumulating a velocity vector in direc...
  4. [ Gradient descent] Learning rate scheduling reduces the step size during training, allowing gradien...
  5. [  Neural networks] A neural network consists of layers of interconnected nodes that learn hierarchi...


---
## Part 2: Dense Retrieval with Static Embeddings

Dense retrieval encodes documents and queries into continuous vector spaces. Unlike BM25, dense embeddings capture **semantic meaning** — "optimization" and "minimization" end up close even without shared tokens.

We use [**potion-base-32M**](https://huggingface.co/minishlab/potion-base-32M), a fast static embedding model from the [model2vec](https://github.com/MinishLab/model2vec) family. It is orders of magnitude faster than transformer-based encoders (no GPU needed) while retaining strong retrieval quality.

The workflow is identical to sparse: retrieve top-K candidates, pass their embeddings to `diversify`.

In [ ]:
# Encode corpus and query with potion-base-32M
# device="cpu" — static embedding models do not benefit from GPU/MPS
model = SentenceTransformer("minishlab/potion-base-32M", device="cpu")
corpus_embeddings = model.encode(corpus, normalize_embeddings=True)       # (25, 512)
query_embedding   = model.encode([query], normalize_embeddings=True)      # (1, 512)

# Cosine similarity scores (vectors are L2-normalised → dot product = cosine sim)
dense_scores = (query_embedding @ corpus_embeddings.T)[0]                 # (25,)

# Select top-K candidates
dense_top_k_indices        = np.argsort(dense_scores)[::-1][:K]
dense_candidate_embeddings = corpus_embeddings[dense_top_k_indices]       # (K, 512)
dense_candidate_scores     = dense_scores[dense_top_k_indices]

print("Top-K dense candidates:")
for rank, (idx, score) in enumerate(zip(dense_top_k_indices, dense_candidate_scores), 1):
    print(f"  {rank}. [{topic_labels[idx]:>17s}] (score={score:.3f}) {corpus[idx][:70]}...")

In [ ]:
# Naive top-5: no diversification — all results from the dominant cluster
naive_dense = diversify(
    embeddings=dense_candidate_embeddings,
    scores=dense_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.0,
)

print("=== Dense — Naive top-5 (diversity=0.0) ===")
for rank, i in enumerate(naive_dense.indices, 1):
    idx = dense_top_k_indices[i]
    print(f"  {rank}. [{topic_labels[idx]:>17s}] {corpus[idx][:80]}...")

In [ ]:
# Moderately diversified top-5 (diversity=0.7)
diverse_dense = diversify(
    embeddings=dense_candidate_embeddings,
    scores=dense_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.7,
)

print("=== Dense — Diversified top-5 (diversity=0.7) ===")
for rank, i in enumerate(diverse_dense.indices, 1):
    idx = dense_top_k_indices[i]
    print(f"  {rank}. [{topic_labels[idx]:>17s}] {corpus[idx][:80]}...")

In [ ]:
# Maximum diversity (diversity=1.0): one result from every topic cluster
# This demonstrates DPP's ability to achieve full topic coverage when needed.
# In practice, values of 0.5–0.8 give the best relevance/diversity trade-off.
diverse_dense_full = diversify(
    embeddings=dense_candidate_embeddings,
    scores=dense_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=1.0,
)

print("=== Dense — Maximum diversity (diversity=1.0) ===")
for rank, i in enumerate(diverse_dense_full.indices, 1):
    idx = dense_top_k_indices[i]
    print(f"  {rank}. [{topic_labels[idx]:>17s}] {corpus[idx][:80]}...")

print()
print("Topic coverage:")
print(f"  Naive (d=0.0) : {sorted(set(topic_labels[dense_top_k_indices[i]] for i in naive_dense.indices))}")
print(f"  Diverse (d=0.7): {sorted(set(topic_labels[dense_top_k_indices[i]] for i in diverse_dense.indices))}")
print(f"  Full (d=1.0)  : {sorted(set(topic_labels[dense_top_k_indices[i]] for i in diverse_dense_full.indices))}")

---
## Part 3: Hybrid Retrieval

Hybrid retrieval fuses BM25 and dense scores before selecting candidates. A simple approach: min-max normalise each distribution to `[0, 1]` then linearly interpolate.

Two separate concerns:
- **Hybrid score** (BM25 + dense) — determines *which* documents make the candidate pool. Fusing both signals improves recall: BM25 surfaces exact keyword matches, dense retrieval surfaces semantic neighbours that share no vocabulary with the query.
- **Dense embeddings** for diversity — determines *how redundant* two candidates are. BM25 term vectors only reflect vocabulary overlap with the query, not passage-to-passage semantic similarity, so they are a poor basis for diversity. Dense embeddings are the right tool here regardless of how candidates were scored.

In [ ]:
# BM25 scores for all 25 documents
all_bm25_scores = retriever.get_scores(query.split())   # (25,)

def min_max_normalize(x: np.ndarray) -> np.ndarray:
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

bm25_norm  = min_max_normalize(all_bm25_scores)
dense_norm = min_max_normalize(dense_scores)   # computed in Part 2

alpha = 0.5
hybrid_scores = alpha * dense_norm + (1 - alpha) * bm25_norm

# Select top-K by hybrid score; use dense embeddings for diversity computation
hybrid_top_k_indices        = np.argsort(hybrid_scores)[::-1][:K]
hybrid_candidate_embeddings = corpus_embeddings[hybrid_top_k_indices]
hybrid_candidate_scores     = hybrid_scores[hybrid_top_k_indices]

print("Top-K hybrid candidates:")
for rank, (idx, score) in enumerate(zip(hybrid_top_k_indices, hybrid_candidate_scores), 1):
    print(f"  {rank}. [{topic_labels[idx]:>17s}] (score={score:.3f}) {corpus[idx][:70]}...")

In [ ]:
# Naive top-5
naive_hybrid = diversify(
    embeddings=hybrid_candidate_embeddings,
    scores=hybrid_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.0,
)

print("=== Hybrid — Naive top-5 (diversity=0.0) ===")
for rank, i in enumerate(naive_hybrid.indices, 1):
    idx = hybrid_top_k_indices[i]
    print(f"  {rank}. [{topic_labels[idx]:>17s}] {corpus[idx][:80]}...")

In [ ]:
# Diversified top-5
diverse_hybrid = diversify(
    embeddings=hybrid_candidate_embeddings,
    scores=hybrid_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.7,
)

print("=== Hybrid — Diversified top-5 (diversity=0.7) ===")
for rank, i in enumerate(diverse_hybrid.indices, 1):
    idx = hybrid_top_k_indices[i]
    print(f"  {rank}. [{topic_labels[idx]:>17s}] {corpus[idx][:80]}...")

---
## Part 4: Comparing the Three Approaches

Side-by-side topic coverage for naive vs diversified across all three retrieval methods.

In [ ]:
def result_topics(result, candidate_indices):
    return [topic_labels[candidate_indices[i]] for i in result.indices]

rows = {
    "BM25 (sparse)" : (naive_bm25,   diverse_bm25,   bm25_candidate_indices),
    "Dense (potion)" : (naive_dense,  diverse_dense,  list(dense_top_k_indices)),
    "Hybrid"         : (naive_hybrid, diverse_hybrid, list(hybrid_top_k_indices)),
}

print(f"{'Method':<18} {'Naive top-5 topics':<45} {'Diversified top-5 topics (d=0.7)'}")
print("-" * 105)
for method, (naive, diverse, cand_idx) in rows.items():
    t_naive  = result_topics(naive,  cand_idx)
    t_diverse = result_topics(diverse, cand_idx)
    print(f"  {method:<16} {str(t_naive):<45} {t_diverse}")

print()
print("Unique topics covered:")
print(f"{'Method':<18} {'Naive':>8} {'Diverse':>10}")
print("-" * 40)
for method, (naive, diverse, cand_idx) in rows.items():
    n = len(set(result_topics(naive, cand_idx)))
    d = len(set(result_topics(diverse, cand_idx)))
    print(f"  {method:<16} {n:>8} {d:>10}")
# Note: BM25 naive already shows 2 unique topics because zero-scoring passages
# are ranked arbitrarily — the sparse section is primarily an API demo.
# The dense and hybrid sections show where diversification has the strongest effect.


---
## Key Takeaways

1. **Pyversity is embedding-agnostic.** DPP only operates on pairwise similarities between vectors — it does not matter whether those vectors came from BM25, a static model, a transformer, or anything else.

2. **`.toarray()` is all you need for sparse vectors.** BM25 score matrices are scipy sparse arrays. One call converts them to a dense NumPy array that pyversity accepts.

3. **Dense embeddings unlock full topic coverage.** With `minishlab/potion-base-32M` and `diversity=1.0`, DPP selects one passage from every topic cluster — turning a redundant result list into a complete overview.

4. **The `diversity` parameter is a dial.** `0.0` = pure relevance ranking. `1.0` = maximum topic coverage. Values of `0.5–0.8` give the best relevance/diversity trade-off for most production use cases.